# 05 · Compare all the models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LekshmiSR/proteins-d1-hands-on/blob/main/notebooks/05_compare_models.ipynb)

Every training notebook (01–04) saved a results file `<model>.json` with its settings, training curves and test metrics. This notebook puts them side by side.

**Where the results come from** (first match wins, per model):
1. your Google Drive folder `protein-dl-results/` (if you set `USE_DRIVE = True` and saved there from notebooks 01–04),
2. the local `results/` folder (if you ran the notebooks in this same session or on your own computer),
3. `results/reference/`: results we produced with the notebooks' default settings, shipped with the repository, so this notebook works even before you train anything.

This notebook trains nothing and runs in under a minute, without a GPU.

## 0. Setup

Run the next cell first. It works in two places:

* **Google Colab**: it downloads this repository (code + helpers) into the Colab machine.
* **Your own computer** (Jupyter / VS Code): it just moves to the repository folder.

No GPU needed here.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/LekshmiSR/proteins-d1-hands-on.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if "google.colab" in sys.modules:                       # running in Colab
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.exists(REPO_DIR):
            subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":      # running locally from notebooks/
    os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from protein_dl import plots, train

USE_DRIVE = False     # True = also look in Google Drive (MyDrive/protein-dl-results)

## 1. Load the results

In [ ]:
MODELS = ["mlp", "cnn", "bilstm", "transformer"]
NICE = {"mlp": "MLP (window)", "cnn": "1D CNN", "bilstm": "BiLSTM", "transformer": "Transformer"}

folders = []
drive_dir = train.get_drive_dir(USE_DRIVE)
if drive_dir:
    folders.append(("your Google Drive", drive_dir))
folders += [("this session (results/)", "results"), ("reference (shipped)", "results/reference")]

results = {}
for m in MODELS:
    for label, folder in folders:
        path = os.path.join(folder, f"{m}.json")
        if os.path.exists(path):
            with open(path) as f:
                results[m] = json.load(f)
            results[m]["source"] = label
            break

for m, r in results.items():
    print(f"{NICE[m]:14s} <- {r['source']:28s} trained on: {r['device']}")
missing = [m for m in MODELS if m not in results]
if missing:
    print("\nNo results found yet for:", missing, "-> run those notebooks first (with SAVE_TO_DRIVE = True in Colab).")

## 2. The summary table

* **Params**: number of trainable weights.
* **Train min**: total training time (depends a lot on the hardware shown above).
* **Val Q3**: best validation accuracy during training.
* **CB513 Q3 / macro F1**: final scores on the independent test set.
* **H / E / C F1**: F1 score of each class on CB513.

In [ ]:
rows = []
for m, r in results.items():
    t = r["test_cb513"]
    rows.append({
        "model": NICE[m],
        "params": r["n_parameters"],
        "epochs": len(r["history"]["epoch"]),
        "train min": round(r["history"]["total_seconds"] / 60, 1),
        "val Q3": round(r["valid"]["q3_accuracy"], 4),
        "CB513 Q3": round(t["q3_accuracy"], 4),
        "CB513 macro F1": round(t["macro_f1"], 4),
        "H F1": round(t["per_class"]["Helix"]["f1"], 3),
        "E F1": round(t["per_class"]["Strand"]["f1"], 3),
        "C F1": round(t["per_class"]["Coil"]["f1"], 3),
    })
table = pd.DataFrame(rows).set_index("model")
table

## 3. Accuracy on CB513

The dashed lines are the two baselines from notebook 00 (no learning of context at all).

In [ ]:
BASELINE_MAJORITY = 0.431      # always predict coil       (notebook 00)
BASELINE_PROPENSITY = 0.489    # best class per amino acid  (notebook 00)

names = list(table.index)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
bars = axes[0].bar(names, table["CB513 Q3"], color="tab:blue")
axes[0].bar_label(bars, fmt="%.3f")
axes[0].axhline(BASELINE_MAJORITY, ls="--", color="grey", label="majority class")
axes[0].axhline(BASELINE_PROPENSITY, ls="--", color="black", label="amino-acid propensity")
axes[0].set_ylim(0.4, max(0.8, table["CB513 Q3"].max() + 0.03))
axes[0].set_ylabel("Q3 accuracy"); axes[0].set_title("CB513 Q3 accuracy"); axes[0].legend(loc="upper left")

width = 0.25
xs = np.arange(len(names))
for k, (cls, col) in enumerate([("H F1", "tab:red"), ("E F1", "tab:blue"), ("C F1", "grey")]):
    axes[1].bar(xs + (k - 1) * width, table[cls], width, label=cls.split()[0], color=col)
axes[1].set_xticks(xs, names); axes[1].set_ylim(0, 1)
axes[1].set_title("F1 per class on CB513"); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Where do the models go wrong?

Confusion matrices (rows = true class, normalised so each row sums to 1). The diagonal is each class's recall.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 3.6))
axes = np.atleast_1d(axes)
for ax, (m, r) in zip(axes, results.items()):
    plots.plot_confusion(r["test_cb513"]["confusion_matrix"], NICE[m], ax=ax)
plt.tight_layout(); plt.show()

## 5. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for m, r in results.items():
    h = r["history"]
    axes[0].plot(h["epoch"], h["val_loss"], "o-", label=NICE[m])
    axes[1].plot(h["epoch"], h["val_q3"], "o-", label=NICE[m])
axes[0].set_title("Validation loss"); axes[0].set_xlabel("epoch")
axes[1].set_title("Validation Q3"); axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Is a bigger or slower model better?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for m, r in results.items():
    q3 = r["test_cb513"]["q3_accuracy"]
    axes[0].scatter(r["n_parameters"], q3, s=80); axes[0].annotate(NICE[m], (r["n_parameters"], q3), xytext=(5, 5), textcoords="offset points")
    sec = r["history"]["total_seconds"] / 60
    axes[1].scatter(sec, q3, s=80); axes[1].annotate(NICE[m], (sec, q3), xytext=(5, 5), textcoords="offset points")
axes[0].set_xlabel("trainable parameters"); axes[0].set_ylabel("CB513 Q3")
axes[1].set_xlabel("training time (minutes, on the device listed above)"); axes[1].set_ylabel("CB513 Q3")
plt.tight_layout(); plt.show()

## 7. Take-home messages

Look at your own numbers above; with the default settings the picture is usually this:

1. **Every network clearly beats the baselines** (≈ 43–49%): the secondary structure of a residue depends on its neighbours, and all four architectures learn to use them.
2. **How the model sees context matters more than its size.** The MLP sees a fixed window with separate weights per position; the CNN shares filters along the chain; the BiLSTM reads the whole chain from both sides; the Transformer can relate any two residues.
3. **Strands (E) are the hardest class** for every model. A β-strand pairs with another strand that can be far away in the sequence, and that long-range information is hard to learn from ~18k single sequences.
4. **Inductive bias matters with limited data.** The Transformer is the most flexible model, but it has to learn locality from scratch. On a dataset of this size it rarely wins against the CNN or BiLSTM. Its strength appears with pretraining on millions of proteins.
5. **The ceiling for single-sequence models is around 70–75% Q3.** Published methods reach 85–88% by adding evolutionary information (multiple sequence alignments) or embeddings from pretrained protein language models.

## Where to go next

* **Hyperparameters:** each notebook ends with exercises. Change one setting at a time and re-run this comparison.
* **8-class task (Q8):** predict the 8 DSSP states instead of 3 (change `encode_q8_as_q3` in `protein_dl/data.py`). Much harder!
* **Hybrid models:** CNN layers followed by a BiLSTM or a Transformer are a strong classic combination.
* **Pretrained protein language model:** use frozen per-residue embeddings from a small **ESM-2** model (e.g. `esm2_t6_8M_UR50D`, 8M parameters, fits a 4 GB GPU) as the input to a small head. This is the modern way to get large gains.